In [2]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.common.by import By

from time import sleep

import os



import numpy as np

from selenium.webdriver.chrome.service import Service as ChromeService




In [3]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'KR FSS' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Read {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running KR FSS Web Scraping Tool v.1.2


In [4]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()





In [8]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        # 'KR FSS 1':'Life Insurance Companies' , #0304

        # 'KR FSS 2': 'Nonlife Insurance Companies', #0102

        # 'KR FSS 3': 'Mutual Savings Banks', #0505

        # #'KR FSS 4': 'Merchant Banking Corporations', 

        # 'KR FSS 5': 'Money Brokerage Companies', #3232

        # 'KR FSS 6': 'Leasing Companies', #1111

        # 'KR FSS 7': 'New Technology Venture Capital', #1010

        # 'KR FSS 8': 'Installment Finance Companies', #0909

        # 'KR FSS 9': 'Credit Card Services Companies', #1212

        # # 'KR FSS 10': 'FINANCIAL HOLDING COMPANIES', 

        # 'KR FSS 11': 'Foreign Bank Branches',  #1515

        # 'KR FSS 12': 'Commercial Banks',  #1618

        # 'KR FSS 13': 'Specialized Banks', #1919

        # 'KR FSS 14': 'Real Estate Investment Trusts',  # 2020

        # 'KR FSS 15': 'Investment Advisory Companies',  # 3838

        # 'KR FSS 16': 'Asset Management Companies', #2424
        # 'KR FSS 17': 'Credit Information Service', 

        'KR FSS 18': 'Futures Companies', 

        'KR FSS 19': 'Securities Companies', 

        'KR FSS 20': 'Credit Rating Companies', 

        'KR FSS 21': 'Fund Rating Companies',

        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



alphabet = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

processdate = now.strftime('%Y-%m-%d')





In [9]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict





In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

# driver.get('about:blank')

driver.get('https://www.fss.or.kr/eng/sprvise/instt/list.do?menuNo=400029')



for k, reg in enumerate(regdict):

	print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} : {regdict[reg]}")

	driver.find_element(By.XPATH, f'//*[@id="selectCate"]/option[contains(text(),"{regdict[reg]}")]').click()


	sleep(2)

	driver.find_element(By.XPATH, f'//*[@id="frm"]//*/div/button[@aria-label="search"]').click() # search button

	sleep(2)



	# alphabet = alphabet[:5]

	for i, letter in enumerate(alphabet):

		driver.find_element(By.XPATH, f'//*[@id="frm"]//*/div/button[contains(text(),"{letter.upper()}")]').click()

		sleep(1)

		soup=BeautifulSoup(driver.page_source, 'html.parser')



		try:

			pages = []

			all_span = soup.find('div', {'class':'pagination-set'}).find_all('span')

			for span in all_span:

				if len(span.attrs)==0 :

					if span.text.isdigit() :

						if span.text not in  pages:

							pages.append(span.text)

			# pages = list(np.unique(pages))

		except:

			pages = []



		# pages = pages[:4]

		for j, page in enumerate(pages):

			try:

				driver.find_element(By.XPATH, f'//*[@id="content"]//*/li/a[@data-pageindex="{page}"]').click()

			except:

				u =9



			sleep(1)

			soup=BeautifulSoup(driver.page_source, 'html.parser')

			tbody = soup.find('tbody')

			trs = tbody.find_all('tr')



			print(f"[INFO] :  - {reg} | alphabet {i+1}/{len(alphabet)} = {letter.upper()} | page {j+1}/{len(pages)} = {page} | {len(trs)} campany" )

			for tr in trs:



				name = tr.find_all('td')[1].text.replace('Name of financial institution', '').strip()

				Website = tr.find_all('td')[2].text.replace('Homepage', '').strip()

				Phone = tr.find_all('td')[3].text.replace('Tel', '').strip()



				# if li.find('a', href=True):

				# 	sqldict['Phone'].append(li.find('a', href=True)['href'].split(':', 1)[1].strip())



				sqldict['Name'].append(name)

				sqldict['Website'].append(Website)

				sqldict['Phone'].append(Phone)

				sqldict['Cntry'].append('KR')

				sqldict['ListProcessDate'].append(processdate)

				sqldict['RegCtry'].append(reg.split()[0])

				sqldict['RegCode'].append(reg.split()[1])

				sqldict['ListCode'].append(reg.split()[2])

				sqldict['RegulationType'].append('Regulated')



			sqldict = bourange_same_length_array(sqldict)



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)


    

[INFO] : Working 1/4 | KR FSS 18 : Futures Companies
[INFO] :  - KR FSS 18 | alphabet 5/26 = E | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 7/26 = G | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 8/26 = H | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 10/26 = J | page 1/1 = 1 | 2 campany
[INFO] :  - KR FSS 18 | alphabet 11/26 = K | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 13/26 = M | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 14/26 = N | page 1/1 = 1 | 2 campany
[INFO] :  - KR FSS 18 | alphabet 18/26 = R | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 19/26 = S | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 20/26 = T | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 18 | alphabet 21/26 = U | page 1/1 = 1 | 1 campany
[INFO] : Working 2/4 | KR FSS 19 : Securities Companies
[INFO] :  - KR FSS 19 | alphabet 1/26 = A | page 1/1 = 1 | 1 campany
[INFO] :  - KR FSS 19 | alphabet 2/26 = B | page 1/1 = 

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_31372\2055007487.py:138: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

: 